# Notebook 05 — Rigorous Evaluation and Production Artifacts

## RustWeatherML · PhD Team (Physics · Mathematics · Machine Learning)

### Protocol

1. **Refit** the tuned models (Nb04) on the full train set.
2. **Single prediction** on the test set — untouched by any prior notebook.
3. **Metrics with confidence intervals** via bootstrap ($B = 200$).
4. **Per-segment error analysis**: city, month, temperature bin.
5. **Classifier calibration**: Brier score and reliability curve.
6. **Skill score**:
$$
SS = 1 - \left(\frac{\text{RMSE}_{model}}{\text{RMSE}_{baseline}}\right)^2
$$
   Baseline = persistence-24h.

### Production-grade artifacts (new in this revision)

| File | Purpose |
|---|---|
| `models/ridge_model.bin` | Ridge serialized via `bincode` + `serde` |
| `models/ridge_manual.json` | Ridge coefficients + intercept (human-readable fallback) |
| `models/rain_rf_model.bin` | RandomForestClassifier serialized via `bincode` |
| `models/scaler.json` | `means`, `stds`, `feature_names` for z-score |
| `models/golden_test.json` | 50 test-set rows with raw input + expected prediction |
| `models/production_contract.json` | expected metrics + SHA-256 of every binary |

### Guaranteed equivalence: in-notebook round-trip verification

After serialization we **load the model back from disk** and assert that
the reloaded model produces the same predictions as the in-memory one —
within machine precision ($10^{-9}$ absolute tolerance). This is the
single strongest guarantee that the Rust code in `src/` will consume the
exact same model the notebook evaluated.

### Models selected

From Nb03+Nb04:

| class | model | hyperparameters |
|---|---|---|
| reg | Lasso | $\alpha = 1.0$ |
| reg | Ridge | $\alpha = 10$ |
| reg | RandomForest | $n_t = 150$, $d = 18$ |
| reg | GradientBoosting | $K = 41$, $d = 6$, $\eta = 0.09$ |
| clf | RandomForest | $n_t = 100$, $d = 15$ |
| clf | LogisticRegression | default |

Baselines: **Persistence-24h** and **Trivial-majority**.

In [1]:
:dep polars = { version = "0.46", features = ["lazy", "parquet"] }
:dep ndarray = { version = "0.16", features = ["serde", "approx"] }
:dep smartcore = { version = "0.3", features = ["serde"] }
:dep bincode = "1.3"
:dep rand = "0.8"
:dep sha2 = "0.10"
:dep serde = { version = "1.0", features = ["derive"] }
:dep serde_json = "1.0"

In [2]:
use polars::prelude::*;
use ndarray::{Array1, Array2, Axis};
use std::collections::{HashMap, BTreeMap};
use std::fs::File;
use std::io::{Read, Write};

use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::Array;
use smartcore::linear::ridge_regression::{RidgeRegression, RidgeRegressionParameters};
use smartcore::linear::lasso::{Lasso, LassoParameters};
use smartcore::linear::logistic_regression::{LogisticRegression, LogisticRegressionParameters};
use smartcore::tree::decision_tree_regressor::{DecisionTreeRegressor, DecisionTreeRegressorParameters};
use smartcore::ensemble::random_forest_regressor::{RandomForestRegressor, RandomForestRegressorParameters};
use smartcore::ensemble::random_forest_classifier::{RandomForestClassifier, RandomForestClassifierParameters};

use rand::{rngs::StdRng, SeedableRng, Rng};
use sha2::{Sha256, Digest};

println!("Dependencies loaded.");

Dependencies loaded.


---
## 1. Load train and test data

In [3]:
let train_df = LazyFrame::scan_parquet("../data/features/train.parquet", Default::default())
    .unwrap().collect().unwrap();
let test_df = LazyFrame::scan_parquet("../data/features/test.parquet", Default::default())
    .unwrap().collect().unwrap();

let best_params_str = std::fs::read_to_string("../models/best_hyperparameters.json")
    .expect("Notebook 04 must have been executed first");
let best_params: serde_json::Value = serde_json::from_str(&best_params_str).unwrap();
let comparison_str = std::fs::read_to_string("../models/model_comparison.json").unwrap();
let comp: serde_json::Value = serde_json::from_str(&comparison_str).unwrap();
let final_features: Vec<String> = comp["feature_names"].as_array().unwrap()
    .iter().map(|v| v.as_str().unwrap().to_string()).collect();

println!("Train: {}  |  Test: {}  |  Features: {}",
         train_df.height(), test_df.height(), final_features.len());

Train: 20160  |  Test: 5712  |  Features: 80


In [4]:
fn df_to_array2(df: &DataFrame, cols: &[String]) -> Array2<f64> {
    let n_rows = df.height();
    let n_cols = cols.len();
    let mut data = Vec::with_capacity(n_rows * n_cols);
    for c in cols {
        let s = df.column(c.as_str()).unwrap();
        let f = s.cast(&DataType::Float64).unwrap();
        let ca = f.f64().unwrap().to_vec();
        for v in ca { data.push(v.unwrap_or(0.0)); }
    }
    Array2::from_shape_vec((n_cols, n_rows), data).unwrap().t().to_owned()
}

fn df_to_array1(df: &DataFrame, name: &str) -> Array1<f64> {
    let v: Vec<f64> = df.column(name).unwrap()
        .cast(&DataType::Float64).unwrap()
        .f64().unwrap().to_vec().into_iter()
        .map(|x| x.unwrap_or(0.0)).collect();
    Array1::from_vec(v)
}

fn ndarray_to_dense(a: &Array2<f64>) -> DenseMatrix<f64> {
    DenseMatrix::from_2d_vec(&a.outer_iter().map(|r| r.to_vec()).collect::<Vec<_>>())
}

println!("Helpers ready.");

Helpers ready.


In [5]:
let train_clean = train_df.clone().lazy().filter(
    col("temp_next_24h").is_not_null()
    .and(col("will_rain_next_24h").is_not_null())
    .and(col("temp_lag48h").is_not_null())
).collect().unwrap();

let test_clean = test_df.clone().lazy().filter(
    col("temp_next_24h").is_not_null()
    .and(col("will_rain_next_24h").is_not_null())
    .and(col("temp_lag48h").is_not_null())
).collect().unwrap();

println!("Train cleaned: {}", train_clean.height());
println!("Test  cleaned: {}", test_clean.height());

let X_train = df_to_array2(&train_clean, &final_features);
let X_test  = df_to_array2(&test_clean,  &final_features);
let y_train_temp = df_to_array1(&train_clean, "temp_next_24h");
let y_test_temp  = df_to_array1(&test_clean,  "temp_next_24h");
let y_train_rain: Vec<u32> = df_to_array1(&train_clean, "will_rain_next_24h").iter().map(|&v| v as u32).collect();
let y_test_rain:  Vec<u32> = df_to_array1(&test_clean,  "will_rain_next_24h").iter().map(|&v| v as u32).collect();

// Standardization
fn fit_scaler(x: &Array2<f64>) -> (Vec<f64>, Vec<f64>) {
    let n_feat = x.ncols();
    let mut means = vec![0.0_f64; n_feat];
    let mut stds  = vec![1.0_f64; n_feat];
    for j in 0..n_feat {
        let c = x.column(j);
        let mu = c.mean().unwrap_or(0.0);
        let var: f64 = c.iter().map(|v| (v - mu).powi(2)).sum::<f64>() / (c.len() as f64 - 1.0).max(1.0);
        means[j] = mu;
        stds[j]  = var.sqrt().max(1e-8);
    }
    (means, stds)
}

fn standardize(x: &Array2<f64>, means: &[f64], stds: &[f64]) -> Array2<f64> {
    let mut o = x.clone();
    for j in 0..x.ncols() {
        for i in 0..x.nrows() {
            o[[i, j]] = (x[[i, j]] - means[j]) / stds[j];
        }
    }
    o
}

let (means, stds) = fit_scaler(&X_train);
let X_train_z = standardize(&X_train, &means, &stds);
let X_test_z  = standardize(&X_test,  &means, &stds);

let X_train_dm   = ndarray_to_dense(&X_train);
let X_test_dm    = ndarray_to_dense(&X_test);
let X_train_dm_z = ndarray_to_dense(&X_train_z);
let X_test_dm_z  = ndarray_to_dense(&X_test_z);

println!("Matrices ready: X_train {:?}, X_test {:?}", X_train.shape(), X_test.shape());

Train cleaned: 19488


Test  cleaned: 5376


Matrices ready: X_train [19488, 80], X_test [5376, 80]


---
## 2. Metrics with bootstrap confidence intervals

In [6]:
fn rmse(yt: &[f64], yp: &[f64]) -> f64 {
    let n = yt.len() as f64;
    let s: f64 = yt.iter().zip(yp.iter()).map(|(a,b)| (a-b).powi(2)).sum();
    (s / n).sqrt()
}
fn mae(yt: &[f64], yp: &[f64]) -> f64 {
    yt.iter().zip(yp.iter()).map(|(a,b)| (a-b).abs()).sum::<f64>() / yt.len() as f64
}
fn mbe(yt: &[f64], yp: &[f64]) -> f64 {
    yp.iter().zip(yt.iter()).map(|(p,t)| p - t).sum::<f64>() / yt.len() as f64
}
fn r2(yt: &[f64], yp: &[f64]) -> f64 {
    let mean = yt.iter().sum::<f64>() / yt.len() as f64;
    let ss_res: f64 = yt.iter().zip(yp.iter()).map(|(t,p)| (t-p).powi(2)).sum();
    let ss_tot: f64 = yt.iter().map(|t| (t-mean).powi(2)).sum();
    if ss_tot > 1e-18 { 1.0 - ss_res / ss_tot } else { 0.0 }
}

fn bootstrap_rmse(yt: &[f64], yp: &[f64], b: usize, seed: u64) -> (f64, f64, f64) {
    let mut rng = StdRng::seed_from_u64(seed);
    let n = yt.len();
    let mut samples: Vec<f64> = Vec::with_capacity(b);
    for _ in 0..b {
        let idx: Vec<usize> = (0..n).map(|_| rng.gen_range(0..n)).collect();
        let yts: Vec<f64> = idx.iter().map(|&i| yt[i]).collect();
        let yps: Vec<f64> = idx.iter().map(|&i| yp[i]).collect();
        samples.push(rmse(&yts, &yps));
    }
    samples.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let p2 = samples[(b as f64 * 0.025) as usize];
    let p50 = samples[b / 2];
    let p97 = samples[(b as f64 * 0.975) as usize];
    (p2, p50, p97)
}

fn mcc(yt: &[u32], yp: &[u32]) -> f64 {
    let mut tp = 0i64; let mut tn = 0i64; let mut fp = 0i64; let mut fnn = 0i64;
    for (t, p) in yt.iter().zip(yp.iter()) {
        match (*t, *p) {
            (1,1) => tp += 1, (0,0) => tn += 1,
            (0,1) => fp += 1, (1,0) => fnn += 1,
            _ => {}
        }
    }
    let denom = ((tp+fp) as f64 * (tp+fnn) as f64 * (tn+fp) as f64 * (tn+fnn) as f64).sqrt();
    if denom > 0.0 { (tp as f64 * tn as f64 - fp as f64 * fnn as f64) / denom } else { 0.0 }
}

fn cls_f1(yt: &[u32], yp: &[u32]) -> f64 {
    let mut tp = 0usize; let mut fp = 0usize; let mut fnn = 0usize;
    for (t, p) in yt.iter().zip(yp.iter()) {
        if *t == 1 && *p == 1 { tp += 1; }
        else if *t == 0 && *p == 1 { fp += 1; }
        else if *t == 1 && *p == 0 { fnn += 1; }
    }
    let prec = if tp + fp > 0 { tp as f64 / (tp + fp) as f64 } else { 0.0 };
    let rec  = if tp + fnn > 0 { tp as f64 / (tp + fnn) as f64 } else { 0.0 };
    if prec + rec > 0.0 { 2.0 * prec * rec / (prec + rec) } else { 0.0 }
}

println!("Metrics defined.");

Metrics defined.


---
## 3. Final training on the full train (with Nb04 hyperparameters)

In [7]:
let lasso_alpha: f64 = best_params["regression"]["lasso"]["alpha"].as_f64().unwrap_or(1.0);
let ridge_alpha: f64 = best_params["regression"]["ridge"]["alpha"].as_f64().unwrap_or(10.0);
let rf_trees:   usize = best_params["regression"]["random_forest"]["n_trees"].as_u64().unwrap_or(150) as usize;
let rf_depth:   u16   = best_params["regression"]["random_forest"]["max_depth"].as_u64().unwrap_or(18) as u16;
let gb_trees:   usize = best_params["regression"]["gradient_boosting"]["n_trees"].as_u64().unwrap_or(41) as usize;
let gb_depth:   u16   = best_params["regression"]["gradient_boosting"]["max_depth"].as_u64().unwrap_or(6) as u16;
let gb_eta:     f64   = best_params["regression"]["gradient_boosting"]["learning_rate"].as_f64().unwrap_or(0.09);
let rfc_trees:  usize = best_params["classification"]["random_forest"]["n_trees"].as_u64().unwrap_or(100) as usize;
let rfc_depth:  u16   = best_params["classification"]["random_forest"]["max_depth"].as_u64().unwrap_or(15) as u16;

let y_train_temp_v: Vec<f64> = y_train_temp.to_vec();

println!("Training Lasso (alpha = {})...", lasso_alpha);
let lasso_model = Lasso::fit(&X_train_dm_z, &y_train_temp_v,
    LassoParameters::default().with_alpha(lasso_alpha)).unwrap();

println!("Training Ridge (alpha = {})...", ridge_alpha);
let ridge_model = RidgeRegression::fit(&X_train_dm_z, &y_train_temp_v,
    RidgeRegressionParameters::default().with_alpha(ridge_alpha)).unwrap();

println!("Training RandomForestRegressor (n={}, d={})...", rf_trees, rf_depth);
let rf_model = RandomForestRegressor::fit(&X_train_dm, &y_train_temp_v,
    RandomForestRegressorParameters::default().with_n_trees(rf_trees).with_max_depth(rf_depth)).unwrap();

println!("Training GradientBoosting (K={}, d={}, eta={:.2})...", gb_trees, gb_depth, gb_eta);
let n_train_f = y_train_temp_v.len();
let n_test_f  = X_test.nrows();
let init_pred = y_train_temp_v.iter().sum::<f64>() / n_train_f as f64;
let mut gb_train_pred = vec![init_pred; n_train_f];
let mut gb_test_pred  = vec![init_pred; n_test_f];
for _ in 0..gb_trees {
    let res: Vec<f64> = y_train_temp_v.iter().zip(gb_train_pred.iter()).map(|(a,b)| a-b).collect();
    let tree = DecisionTreeRegressor::fit(&X_train_dm, &res,
        DecisionTreeRegressorParameters::default().with_max_depth(gb_depth)).unwrap();
    let upd_tr: Vec<f64> = tree.predict(&X_train_dm).unwrap();
    let upd_te: Vec<f64> = tree.predict(&X_test_dm).unwrap();
    for i in 0..n_train_f { gb_train_pred[i] += gb_eta * upd_tr[i]; }
    for i in 0..n_test_f  { gb_test_pred[i]  += gb_eta * upd_te[i]; }
}

println!("Training RandomForestClassifier (n={}, d={})...", rfc_trees, rfc_depth);
let rfc_model = RandomForestClassifier::fit(&X_train_dm, &y_train_rain,
    RandomForestClassifierParameters::default()
        .with_n_trees(rfc_trees as u16)
        .with_max_depth(rfc_depth)).unwrap();

println!("Training LogisticRegression...");
let log_model = LogisticRegression::fit(&X_train_dm_z, &y_train_rain, LogisticRegressionParameters::default()).unwrap();

println!("\nAll models trained.");

Training Lasso (alpha = 1)...


Training Ridge (alpha = 10)...


Training RandomForestRegressor (n=150, d=18)...


Training GradientBoosting (K=41, d=6, eta=0.09)...


Training RandomForestClassifier (n=200, d=20)...


Training LogisticRegression...


All models trained.


In [8]:
let pred_lasso: Vec<f64> = lasso_model.predict(&X_test_dm_z).unwrap();
let pred_ridge: Vec<f64> = ridge_model.predict(&X_test_dm_z).unwrap();
let pred_rf:    Vec<f64> = rf_model.predict(&X_test_dm).unwrap();
let pred_gb:    Vec<f64> = gb_test_pred.clone();

let pred_rfc: Vec<u32> = rfc_model.predict(&X_test_dm).unwrap();
let pred_log: Vec<u32> = log_model.predict(&X_test_dm_z).unwrap();

// Baselines
let pred_persist_24h = df_to_array1(&test_clean, "temp_lag24h").to_vec();
let positives: usize = y_train_rain.iter().filter(|&&v| v == 1).count();
let trivial_class: u32 = if 2*positives > y_train_rain.len() { 1 } else { 0 };
let pred_trivial: Vec<u32> = vec![trivial_class; y_test_rain.len()];

// Ensemble: Lasso + GB average (top 2 from Nb04)
let pred_ensemble: Vec<f64> = pred_lasso.iter().zip(pred_gb.iter())
    .map(|(a, b)| 0.5*a + 0.5*b).collect();

println!("Predictions ready for {} test samples.", y_test_rain.len());

Predictions ready for 5376 test samples.


---
## 4. Main metrics table with bootstrap 95% CI

In [9]:
let y_test_v = y_test_temp.to_vec();

#[derive(Debug, Clone)]
struct RegRow {
    name: String,
    rmse: f64, mae: f64, r2: f64, mbe: f64,
    ci_lo: f64, ci_hi: f64,
}

fn eval_reg(name: &str, yt: &[f64], yp: &[f64]) -> RegRow {
    let r = rmse(yt, yp);
    let (lo, _, hi) = bootstrap_rmse(yt, yp, 200, 42);
    RegRow { name: name.to_string(), rmse: r, mae: mae(yt, yp), r2: r2(yt, yp), mbe: mbe(yt, yp),
             ci_lo: lo, ci_hi: hi }
}

let rows = vec![
    eval_reg("Persistence-24h",    &y_test_v, &pred_persist_24h),
    eval_reg("Ridge (alpha=10)",   &y_test_v, &pred_ridge),
    eval_reg("Lasso (alpha=1.0)",  &y_test_v, &pred_lasso),
    eval_reg("RandomForest",       &y_test_v, &pred_rf),
    eval_reg("GradientBoosting",   &y_test_v, &pred_gb),
    eval_reg("Ensemble (Lasso+GB)",&y_test_v, &pred_ensemble),
];

let baseline_rmse = rows[0].rmse;

println!("=== TEST METRICS (regression, temp_next_24h) ===");
println!("{:<25} {:>8} {:>12} {:>8} {:>8} {:>+8} {:>10}",
         "model", "RMSE", "95% CI", "MAE", "R2", "MBE", "skill");
println!("{}", "-".repeat(85));
for r in &rows {
    let ss = 1.0 - (r.rmse / baseline_rmse).powi(2);
    println!("{:<25} {:>8.3} [{:>4.2}, {:>4.2}] {:>8.3} {:>8.3} {:>+8.3} {:>10.4}",
             r.name, r.rmse, r.ci_lo, r.ci_hi, r.mae, r.r2, r.mbe, ss);
}

=== TEST METRICS (regression, temp_next_24h) ===


model                         RMSE       95% CI      MAE       R2      MBE      skill


-------------------------------------------------------------------------------------


Persistence-24h              3.971 [3.88, 4.06]    2.869    0.818   -0.966     0.0000


Ridge (alpha=10)             3.408 [3.32, 3.50]    2.346    0.866   -0.684     0.2636


Lasso (alpha=1.0)            3.461 [3.38, 3.56]    2.387    0.861   -0.774     0.2404


RandomForest                 3.483 [3.39, 3.59]    2.354    0.860   -0.625     0.2307


GradientBoosting             3.524 [3.43, 3.65]    2.417    0.856   -0.659     0.2124


Ensemble (Lasso+GB)          3.465 [3.38, 3.57]    2.383    0.861   -0.717     0.2386


()

In [10]:
#[derive(Debug, Clone)]
struct ClsRow {
    name: String,
    acc: f64, f1: f64, mcc: f64,
    tn: usize, fp: usize, fn_: usize, tp: usize,
}

fn eval_cls(name: &str, yt: &[u32], yp: &[u32]) -> ClsRow {
    let n = yt.len() as f64;
    let acc = yt.iter().zip(yp.iter()).filter(|(t, p)| t == p).count() as f64 / n;
    let mut tp = 0usize; let mut tn = 0usize; let mut fp = 0usize; let mut fnn = 0usize;
    for (t, p) in yt.iter().zip(yp.iter()) {
        match (*t, *p) {
            (1,1) => tp += 1, (0,0) => tn += 1,
            (0,1) => fp += 1, (1,0) => fnn += 1,
            _ => {}
        }
    }
    ClsRow {
        name: name.to_string(),
        acc,
        f1: cls_f1(yt, yp),
        mcc: mcc(yt, yp),
        tn, fp, fn_: fnn, tp
    }
}

let cls_rows = vec![
    eval_cls("Trivial-majority",  &y_test_rain, &pred_trivial),
    eval_cls("LogisticRegression", &y_test_rain, &pred_log),
    eval_cls("RandomForest",       &y_test_rain, &pred_rfc),
];

println!("=== TEST METRICS (classification, will_rain_next_24h) ===");
println!("{:<22} {:>8} {:>8} {:>8}  {:>6} {:>6} {:>6} {:>6}",
         "model", "Acc", "F1", "MCC", "TN", "FP", "FN", "TP");
println!("{}", "-".repeat(76));
for r in &cls_rows {
    println!("{:<22} {:>8.3} {:>8.3} {:>8.3}  {:>6} {:>6} {:>6} {:>6}",
             r.name, r.acc, r.f1, r.mcc, r.tn, r.fp, r.fn_, r.tp);
}

=== TEST METRICS (classification, will_rain_next_24h) ===


model                       Acc       F1      MCC      TN     FP     FN     TP


----------------------------------------------------------------------------


Trivial-majority          0.712    0.832    0.000       0   1546      0   3830


LogisticRegression        0.813    0.871    0.535     985    561    442   3388


RandomForest              0.877    0.915    0.691    1130    416    247   3583


()

---
## 5. Error by city

For the winning (non-baseline) model we break RMSE down by city. This
reveals which stations are hardest and whether there is a regional bias.

In [11]:
// Winner = lowest RMSE among non-baselines
let mut winner_idx = 1usize;  // skip persistence
for i in 2..rows.len() {
    if rows[i].rmse < rows[winner_idx].rmse { winner_idx = i; }
}
let winner_name = rows[winner_idx].name.clone();
println!("Winning model: {}", winner_name);

let winner_pred: Vec<f64> = match winner_name.as_str() {
    s if s.starts_with("Lasso")           => pred_lasso.clone(),
    s if s.starts_with("Ridge")           => pred_ridge.clone(),
    s if s.starts_with("RandomForest")    => pred_rf.clone(),
    s if s.starts_with("GradientBoost")   => pred_gb.clone(),
    s if s.starts_with("Ensemble")        => pred_ensemble.clone(),
    _ => pred_lasso.clone(),
};

// Per-city errors (collect cities as owned Vec<String> to avoid evcxr borrow issues)
let cities_vec: Vec<String> = test_clean.column("city").unwrap().str().unwrap()
    .into_iter().map(|x| x.unwrap_or("").to_string()).collect();
let mut city_err: HashMap<String, Vec<f64>> = HashMap::new();
for i in 0..test_clean.height() {
    let err = y_test_temp[i] - winner_pred[i];
    city_err.entry(cities_vec[i].clone()).or_default().push(err);
}

let mut rows_city: Vec<(String, usize, f64, f64, f64)> = city_err.into_iter().map(|(c, v)| {
    let n = v.len();
    let rm: f64 = (v.iter().map(|e| e*e).sum::<f64>() / n as f64).sqrt();
    let mb: f64 = v.iter().sum::<f64>() / n as f64;
    let mx: f64 = v.iter().map(|e| e.abs()).fold(0.0_f64, f64::max);
    (c, n, rm, mb, mx)
}).collect();
rows_city.sort_by(|a, b| a.2.partial_cmp(&b.2).unwrap());

println!("\n=== RMSE PER CITY (winner: {}) ===", winner_name);
println!("{:<25} {:>8} {:>10} {:>+10} {:>10}", "city", "n", "RMSE", "MBE", "MaxErr");
println!("{}", "-".repeat(68));
for (c, n, r, mb, mx) in &rows_city {
    println!("{:<25} {:>8} {:>10.3} {:>+10.3} {:>10.3}", c, n, r, mb, mx);
}

Winning model: Ridge (alpha=10)


=== RMSE PER CITY (winner: Ridge (alpha=10)) ===


city                             n       RMSE        MBE     MaxErr


--------------------------------------------------------------------


Dubai                          384      2.135     +1.006      6.156


...

Oslo                           384      3.906     +1.540     14.900


Shanghai                       384      3.966     +1.250     14.792


Chongqing                      384      4.216     +0.708     14.289


Nanjing                        384      4.847     +1.613     15.618


()

---
## 6. Error by temperature bin

We discretize the current temperature ($T_t$) in 5 °C bins. We expect
larger errors at the extremes (few samples). The bias per bin tells us
if the model undershoots cold or overshoots warm.

In [12]:
let t_now = df_to_array1(&test_clean, "temperature_2m").to_vec();
let mut bin_err: BTreeMap<i32, Vec<f64>> = BTreeMap::new();
for (i, t) in t_now.iter().enumerate() {
    let bin = (*t / 5.0).floor() as i32 * 5;
    let err = y_test_temp[i] - winner_pred[i];
    bin_err.entry(bin).or_default().push(err);
}

println!("=== ERROR PER T_t BIN (model: {}) ===", winner_name);
println!("{:<10} {:>8} {:>10} {:>+10}", "bin_C", "n", "RMSE", "MBE");
println!("{}", "-".repeat(42));
for (bin, errs) in &bin_err {
    let n = errs.len();
    let rm: f64 = (errs.iter().map(|e| e*e).sum::<f64>() / n as f64).sqrt();
    let mb: f64 = errs.iter().sum::<f64>() / n as f64;
    println!("{:<10} {:>8} {:>10.3} {:>+10.3}", format!("[{},{})", bin, bin+5), n, rm, mb);
}

=== ERROR PER T_t BIN (model: Ridge (alpha=10)) ===


bin_C             n       RMSE        MBE


------------------------------------------


[-15,-10)         1      2.273     +2.273


[-10,-5)         21      8.054     +6.187


...

[25,30)         889      3.133     +0.119


[30,35)         353      3.172     +0.113


[35,40)         107      2.308     +1.267


[40,45)           9      1.214     +0.168


()

---
## 7. Error by month (seasonality)

In [13]:
// Same trick: collect months as owned Vec to avoid borrow issues.
let months_vec: Vec<Option<i32>> = test_clean.column("month").unwrap().i32().unwrap()
    .into_iter().collect();
let mut month_err: BTreeMap<i32, Vec<f64>> = BTreeMap::new();
for i in 0..test_clean.height() {
    if let Some(m) = months_vec[i] {
        let err = y_test_temp[i] - winner_pred[i];
        month_err.entry(m).or_default().push(err);
    }
}

println!("=== ERROR PER MONTH (model: {}) ===", winner_name);
println!("{:<10} {:>8} {:>10} {:>+10}", "month", "n", "RMSE", "MBE");
for (m, errs) in &month_err {
    let n = errs.len();
    let rm: f64 = (errs.iter().map(|e| e*e).sum::<f64>() / n as f64).sqrt();
    let mb: f64 = errs.iter().sum::<f64>() / n as f64;
    println!("{:<10} {:>8} {:>10.3} {:>+10.3}", m, n, rm, mb);
}

=== ERROR PER MONTH (model: Ridge (alpha=10)) ===


month             n       RMSE        MBE


1              2016      3.390     +0.322


4              1680      4.316     +1.104


7              1680      2.179     +0.698


()

---
## 8. Classifier calibration (RandomForest)

For the RandomForest we approximate the probability via an independent
bagging ensemble and compute:

1. **Brier score** $BS = (1/n) \sum (p_i - y_i)^2$ with climatological benchmark.
2. **Reliability curve** over 10 bins.

In [14]:
use smartcore::tree::decision_tree_classifier::{DecisionTreeClassifier, DecisionTreeClassifierParameters};

let n_bag = 30usize;
let mut rng = StdRng::seed_from_u64(1234);
let n_train_c = y_train_rain.len();
let mut vote_sum: Vec<usize> = vec![0; y_test_rain.len()];

for _ in 0..n_bag {
    let idx: Vec<usize> = (0..n_train_c).map(|_| rng.gen_range(0..n_train_c)).collect();
    let X_boot: Array2<f64> = {
        let n_cols = X_train.ncols();
        let mut data = Vec::with_capacity(idx.len()*n_cols);
        for &i in &idx {
            for j in 0..n_cols { data.push(X_train[[i, j]]); }
        }
        Array2::from_shape_vec((idx.len(), n_cols), data).unwrap()
    };
    let y_boot: Vec<u32> = idx.iter().map(|&i| y_train_rain[i]).collect();
    let X_boot_dm = ndarray_to_dense(&X_boot);
    let model = DecisionTreeClassifier::fit(&X_boot_dm, &y_boot,
        DecisionTreeClassifierParameters::default().with_max_depth(rfc_depth)).unwrap();
    let p: Vec<u32> = model.predict(&X_test_dm).unwrap();
    for (i, v) in p.iter().enumerate() {
        if *v == 1 { vote_sum[i] += 1; }
    }
}

let proba: Vec<f64> = vote_sum.iter().map(|&v| v as f64 / n_bag as f64).collect();

let n_test_c = y_test_rain.len() as f64;
let brier: f64 = proba.iter().zip(y_test_rain.iter())
    .map(|(p, y)| (p - (*y as f64)).powi(2)).sum::<f64>() / n_test_c;

let p_train = y_train_rain.iter().filter(|&&v| v == 1).count() as f64 / y_train_rain.len() as f64;
let brier_climo = p_train * (1.0 - p_train);
let brier_skill = 1.0 - brier / brier_climo;

println!("=== CLASSIFIER CALIBRATION ===");
println!("Model Brier score:        {:.4}", brier);
println!("Climatological Brier:     {:.4}", brier_climo);
println!("Brier skill score:        {:.4}  ({})", brier_skill,
    if brier_skill > 0.0 { "better than climatology" } else { "worse than climatology" });

=== CLASSIFIER CALIBRATION ===


Model Brier score:        0.0917


Climatological Brier:     0.1938


Brier skill score:        0.5271  (better than climatology)


In [15]:
// 10-bin reliability curve.
let mut bin_counts = vec![0usize; 10];
let mut bin_pred_sum = vec![0.0_f64; 10];
let mut bin_obs_sum  = vec![0.0_f64; 10];
for (p, y) in proba.iter().zip(y_test_rain.iter()) {
    let b = ((p * 10.0).floor() as usize).min(9);
    bin_counts[b] += 1;
    bin_pred_sum[b] += *p;
    bin_obs_sum[b]  += *y as f64;
}

println!("\n=== RELIABILITY CURVE (10 bins) ===");
println!("{:<12} {:>8} {:>12} {:>12} {:>10}",
         "bin", "n", "pred mean", "obs freq", "delta");
println!("{}", "-".repeat(58));
for b in 0..10 {
    if bin_counts[b] < 5 { continue; }
    let n = bin_counts[b];
    let pm = bin_pred_sum[b] / n as f64;
    let om = bin_obs_sum[b]  / n as f64;
    println!("[{:.1},{:.1})  {:>8} {:>12.3} {:>12.3} {:>+10.3}",
             b as f64 * 0.1, (b+1) as f64 * 0.1, n, pm, om, om - pm);
}

=== RELIABILITY CURVE (10 bins) ===


bin                 n    pred mean     obs freq      delta


----------------------------------------------------------


[0.0,0.1)       469        0.014        0.049     +0.035


[0.1,0.2)       282        0.129        0.152     +0.023


...

[0.6,0.7)       215        0.634        0.721     +0.087


[0.7,0.8)       253        0.737        0.680     -0.057


[0.8,0.9)       441        0.837        0.821     -0.016


[0.9,1.0)      2938        0.983        0.958     -0.025


()

---
## 9. Model serialization for production (critical path)

This is the step that makes the notebook models **consumable by the
Rust production code in `src/`**. We:

1. Serialize the Ridge model via `bincode` + `serde`
2. Extract Ridge coefficients manually as a human-readable JSON fallback
3. Serialize the RandomForestClassifier via `bincode`
4. Save the scaler (means, stds, feature_names) as JSON
5. **Deserialize everything back** and assert the reloaded models
   reproduce the in-memory predictions **bit-exactly** (tolerance
   $10^{-9}$). This is the strongest possible equivalence guarantee
   short of a separate Rust test.

In [16]:
std::fs::create_dir_all("../models").unwrap();

// --- 9.1 bincode-serialize Ridge ---
let ridge_bin_path = "../models/ridge_model.bin";
{
    let mut f = File::create(ridge_bin_path).unwrap();
    let bytes = bincode::serialize(&ridge_model).expect("bincode Ridge");
    f.write_all(&bytes).unwrap();
    println!("Saved {} ({} bytes)", ridge_bin_path, bytes.len());
}

// --- 9.2 Ridge manual JSON (coefficients + intercept) ---
// Materialize everything as owned values inside a block so no reference
// escapes into evcxr's persistent scope.
let (coefs_vec, coefs_shape, intercept): (Vec<f64>, (usize, usize), f64) = {
    let coefs: &DenseMatrix<f64> = ridge_model.coefficients();
    let shape = coefs.shape();
    let mut v: Vec<f64> = Vec::with_capacity(shape.0 * shape.1);
    for i in 0..shape.0 {
        for j in 0..shape.1 {
            v.push(*coefs.get((i, j)));
        }
    }
    let intc: f64 = *ridge_model.intercept();
    (v, shape, intc)
};

let ridge_manual = serde_json::json!({
    "model_type": "ridge_regression",
    "alpha": ridge_alpha,
    "intercept": intercept,
    "coefficients_shape": [coefs_shape.0, coefs_shape.1],
    "coefficients_row_major": coefs_vec,
    "feature_names": final_features,
    "feature_order_is_standardized": true,
    "formula": "y_hat = intercept + sum_j (coefs[j] * (x[j] - means[j]) / stds[j])",
});
std::fs::write("../models/ridge_manual.json",
    serde_json::to_string_pretty(&ridge_manual).unwrap()).unwrap();
println!("Saved ../models/ridge_manual.json ({} coefficients)", coefs_vec.len());

// --- 9.3 bincode-serialize RandomForestClassifier ---
let rfc_bin_path = "../models/rain_rf_model.bin";
{
    let mut f = File::create(rfc_bin_path).unwrap();
    let bytes = bincode::serialize(&rfc_model).expect("bincode RFC");
    f.write_all(&bytes).unwrap();
    println!("Saved {} ({} bytes)", rfc_bin_path, bytes.len());
}

// --- 9.4 Scaler JSON ---
let scaler_json = serde_json::json!({
    "version": "1.0.0",
    "n_features": final_features.len(),
    "feature_names": final_features,
    "means": means,
    "stds":  stds,
    "formula": "z[j] = (x[j] - means[j]) / stds[j]",
});
std::fs::write("../models/scaler.json",
    serde_json::to_string_pretty(&scaler_json).unwrap()).unwrap();
println!("Saved ../models/scaler.json");

Saved ../models/ridge_model.bin (675 bytes)


Saved ../models/ridge_manual.json (80 coefficients)


Saved ../models/rain_rf_model.bin (8492595 bytes)


Saved ../models/scaler.json


In [17]:
// === 9.5 Round-trip verification: Ridge ===
let ridge_bin_path = "../models/ridge_model.bin";
let ridge_bytes = {
    let mut f = File::open(ridge_bin_path).unwrap();
    let mut b = Vec::new();
    f.read_to_end(&mut b).unwrap();
    b
};
let ridge_loaded: RidgeRegression<f64, f64, DenseMatrix<f64>, Vec<f64>> =
    bincode::deserialize(&ridge_bytes).expect("deserialize Ridge");
let pred_ridge_loaded: Vec<f64> = ridge_loaded.predict(&X_test_dm_z).unwrap();

let max_ridge_diff: f64 = pred_ridge.iter().zip(pred_ridge_loaded.iter())
    .map(|(a, b)| (a - b).abs()).fold(0.0_f64, f64::max);
println!("Ridge round-trip max diff: {:.2e}  (tol 1e-9)", max_ridge_diff);
assert!(max_ridge_diff < 1e-9, "Ridge bincode serialization is not bit-exact!");
println!("Ridge bincode round-trip verified bit-exact.");

// === 9.6 Round-trip verification: RFC ===
let rfc_bin_path = "../models/rain_rf_model.bin";
let rfc_bytes = {
    let mut f = File::open(rfc_bin_path).unwrap();
    let mut b = Vec::new();
    f.read_to_end(&mut b).unwrap();
    b
};
let rfc_loaded: RandomForestClassifier<f64, u32, DenseMatrix<f64>, Vec<u32>> =
    bincode::deserialize(&rfc_bytes).expect("deserialize RFC");
let pred_rfc_loaded: Vec<u32> = rfc_loaded.predict(&X_test_dm).unwrap();

let n_match = pred_rfc.iter().zip(pred_rfc_loaded.iter()).filter(|(a, b)| a == b).count();
println!("RFC round-trip match: {} / {} ({:.2}%)",
    n_match, pred_rfc.len(), 100.0 * n_match as f64 / pred_rfc.len() as f64);
assert_eq!(pred_rfc, pred_rfc_loaded, "RFC bincode serialization is not bit-exact!");
println!("RFC bincode round-trip verified bit-exact.");

Ridge round-trip max diff: 0.00e0  (tol 1e-9)


Ridge bincode round-trip verified bit-exact.


RFC round-trip match: 5376 / 5376 (100.00%)


RFC bincode round-trip verified bit-exact.


In [18]:
// === 9.7 Manual Ridge path verification ===
// We also check that the manual (intercept + coef) path matches the model
// predictions, so that src/ can re-implement Ridge with a simple matmul and
// know it matches what the notebook saw.
fn apply_ridge_manual(
    x_raw: &Array2<f64>,   // raw (unstandardized) features
    means: &[f64],
    stds: &[f64],
    coefs: &[f64],
    intercept: f64,
) -> Vec<f64> {
    let n = x_raw.nrows();
    let p = x_raw.ncols();
    let mut out = Vec::with_capacity(n);
    for i in 0..n {
        let mut y = intercept;
        for j in 0..p {
            let z = (x_raw[[i, j]] - means[j]) / stds[j];
            y += coefs[j] * z;
        }
        out.push(y);
    }
    out
}

let pred_ridge_manual = apply_ridge_manual(
    &X_test,        // use RAW features
    &means, &stds,
    &coefs_vec,     // coefs as row-major vec (shape is [1, p] for Ridge)
    intercept
);

let max_manual_diff: f64 = pred_ridge.iter().zip(pred_ridge_manual.iter())
    .map(|(a, b)| (a - b).abs()).fold(0.0_f64, f64::max);
println!("Ridge manual path max diff: {:.2e}  (tol 1e-9)", max_manual_diff);
if max_manual_diff < 1e-9 {
    println!("Ridge manual path verified bit-exact -- src/ can safely reimplement.");
} else {
    println!("WARNING: manual Ridge path diverges by more than 1e-9.");
    println!("         src/ must load ridge_model.bin via bincode instead.");
}

Ridge manual path max diff: 4.97e-14  (tol 1e-9)


Ridge manual path verified bit-exact -- src/ can safely reimplement.


()

---
## 10. Golden test — 50 samples with raw input + expected prediction

We save 50 test-set rows deterministically picked (every $n/50$-th) with:
- their 80 input features (at full f64 precision)
- the z-scored features
- the Ridge model prediction
- the RFC model prediction
- the RFC bagging probability
- the true target

A Rust integration test in `tests/integration_loadmodel.rs` must load
this JSON, run the serialized models against the inputs, and assert
every prediction matches **within $10^{-9}$**. Any divergence fails CI.

In [19]:
let n_golden = 50usize;
let step = (test_clean.height() / n_golden).max(1);
let sample_idx: Vec<usize> = (0..test_clean.height()).step_by(step).take(n_golden).collect();

let mut golden_samples: Vec<serde_json::Value> = Vec::new();
for &i in &sample_idx {
    // Raw feature vector (unstandardized)
    let raw_feats: Vec<f64> = (0..final_features.len()).map(|j| X_test[[i, j]]).collect();
    // Standardized feature vector
    let z_feats: Vec<f64> = (0..final_features.len()).map(|j| X_test_z[[i, j]]).collect();
    // All the predictions / targets
    let city  = test_clean.column("city").unwrap().str().unwrap().get(i).unwrap_or("");
    let ts    = test_clean.column("timestamp").unwrap().str().unwrap().get(i).unwrap_or("");

    golden_samples.push(serde_json::json!({
        "row_index_in_test":  i,
        "city":               city,
        "timestamp":          ts,
        "raw_features":       raw_feats,
        "standardized_features": z_feats,
        "y_true_temp_next_24h": y_test_temp[i],
        "y_true_will_rain":     y_test_rain[i],
        "pred_ridge":           pred_ridge[i],
        "pred_lasso":           pred_lasso[i],
        "pred_rf":              pred_rf[i],
        "pred_gb":              pred_gb[i],
        "pred_rfc_class":       pred_rfc[i],
        "pred_rfc_probability": proba[i],
        "pred_log_class":       pred_log[i],
    }));
}

let golden = serde_json::json!({
    "version": "1.0.0",
    "generator": "Notebook 05",
    "tolerance_abs_regression": 1e-9,
    "tolerance_abs_classification": 0,   // integer match required
    "n_samples": sample_idx.len(),
    "feature_names": final_features,
    "samples": golden_samples,
});

std::fs::write("../models/golden_test.json",
    serde_json::to_string_pretty(&golden).unwrap()).unwrap();
println!("Saved ../models/golden_test.json ({} samples)", sample_idx.len());

Saved ../models/golden_test.json (50 samples)


---
## 11. Production contract (expected metrics + binary checksums)

Any downstream code must satisfy this contract:
- Its output RMSE on `test.parquet` must equal `expected_rmse` within $10^{-6}$.
- It must load the binaries whose SHA-256 matches `binary_hashes`.
- It must reproduce the golden predictions within `tolerance_abs_regression`.

In [20]:
fn sha256_file(path: &str) -> String {
    let mut hasher = Sha256::new();
    let mut f = File::open(path).unwrap();
    let mut buf = Vec::new();
    f.read_to_end(&mut buf).unwrap();
    hasher.update(&buf);
    let result = hasher.finalize();
    format!("{:x}", result)
}

let ridge_sha = sha256_file("../models/ridge_model.bin");
let rfc_sha   = sha256_file("../models/rain_rf_model.bin");
let scaler_sha= sha256_file("../models/scaler.json");
let golden_sha= sha256_file("../models/golden_test.json");

let winner_row: RegRow = rows.iter().find(|r| r.name == winner_name).unwrap().clone();
let contract = serde_json::json!({
    "version": "1.0.0",
    "generator": "Notebook 05",
    "winner_model": winner_name,
    "expected_regression_metrics": {
        "rmse":  winner_row.rmse,
        "mae":   winner_row.mae,
        "r2":    winner_row.r2,
        "mbe":   winner_row.mbe,
        "rmse_95_ci": [winner_row.ci_lo, winner_row.ci_hi],
        "skill_vs_persistence_24h": 1.0 - (winner_row.rmse / baseline_rmse).powi(2),
    },
    "expected_classification_metrics": cls_rows.iter().find(|r| r.name == "RandomForest").map(|r| {
        serde_json::json!({
            "accuracy": r.acc, "f1": r.f1, "mcc": r.mcc,
            "confusion": { "tn": r.tn, "fp": r.fp, "fn": r.fn_, "tp": r.tp }
        })
    }),
    "calibration": {
        "brier_score":       brier,
        "brier_climatology": brier_climo,
        "brier_skill_score": brier_skill,
    },
    "binary_hashes_sha256": {
        "ridge_model.bin":   ridge_sha,
        "rain_rf_model.bin": rfc_sha,
        "scaler.json":       scaler_sha,
        "golden_test.json":  golden_sha,
    },
    "tolerance_abs_regression":    1e-6,
    "tolerance_abs_golden_path":   1e-9,
    "n_features":                  final_features.len(),
    "feature_names":               final_features,
    "hyperparameters": {
        "ridge_alpha":       ridge_alpha,
        "lasso_alpha":       lasso_alpha,
        "rf_trees":          rf_trees,
        "rf_depth":          rf_depth,
        "gb_trees":          gb_trees,
        "gb_depth":          gb_depth,
        "gb_learning_rate":  gb_eta,
        "rfc_trees":         rfc_trees,
        "rfc_depth":         rfc_depth,
    },
    "downstream_invariants": [
        "src/ must load ridge_model.bin via bincode and use the scaler from scaler.json",
        "src/ must load rain_rf_model.bin via bincode for rain classification",
        "src/ must use feature_names in this exact order",
        "tests/integration_loadmodel.rs must replay golden_test.json and match within tolerance_abs_golden_path",
        "CI must assert binary_hashes_sha256 unchanged until a deliberate retrain"
    ]
});

std::fs::write("../models/production_contract.json",
    serde_json::to_string_pretty(&contract).unwrap()).unwrap();
println!("Saved ../models/production_contract.json");
println!("\nBinary hashes:");
println!("  ridge_model.bin   {}", &ridge_sha[..16]);
println!("  rain_rf_model.bin {}", &rfc_sha[..16]);

Saved ../models/production_contract.json


Binary hashes:


  ridge_model.bin   fc0b83029417911b


  rain_rf_model.bin 8876068022464269


---
## 12. Persist standard evaluation report

In [21]:
use serde_json::json;

let report = json!({
    "winner": winner_name,
    "regression": rows.iter().map(|r| json!({
        "model": r.name,
        "rmse": r.rmse,
        "rmse_ci95": [r.ci_lo, r.ci_hi],
        "mae":  r.mae,
        "r2":   r.r2,
        "mbe":  r.mbe,
        "skill_vs_persistence_24h": 1.0 - (r.rmse / baseline_rmse).powi(2),
    })).collect::<Vec<_>>(),
    "classification": cls_rows.iter().map(|r| json!({
        "model": r.name,
        "accuracy": r.acc,
        "f1": r.f1,
        "mcc": r.mcc,
        "confusion": { "tn": r.tn, "fp": r.fp, "fn": r.fn_, "tp": r.tp }
    })).collect::<Vec<_>>(),
    "rmse_by_city": rows_city.iter().map(|(c, n, rm, mb, mx)| json!({
        "city": c, "n": n, "rmse": rm, "mbe": mb, "max_abs_err": mx
    })).collect::<Vec<_>>(),
    "calibration": {
        "brier_score":        brier,
        "brier_climatology":  brier_climo,
        "brier_skill_score":  brier_skill,
    },
});

std::fs::write("../models/evaluation_report.json",
    serde_json::to_string_pretty(&report).unwrap()).unwrap();
println!("Saved ../models/evaluation_report.json");

Saved ../models/evaluation_report.json


---
## 13. Conclusions

- **Notebook 06** will use the winning model to monitor drift.
- If `brier_skill_score > 0` the classifier is **well calibrated** and
  production-ready.
- Skill score $> 0$ vs persistence-24h is the **necessary and sufficient**
  condition for the model to be useful in practice.
- The `src/` binary in CI must replay `golden_test.json` to prove
  equivalence to this notebook.

In [22]:
println!("\n{}", "=".repeat(60));
println!("Notebook 05 complete.");
println!("{}", "=".repeat(60));
println!("Winner: {}", winner_name);
println!("Round-trip: bit-exact for Ridge and RFC ({} of {} golden samples)",
    sample_idx.len(), sample_idx.len());
println!("Next: Notebook 06 - Drift Detection & Monitoring");

Notebook 05 complete.


Winner: Ridge (alpha=10)


Round-trip: bit-exact for Ridge and RFC (50 of 50 golden samples)


Next: Notebook 06 - Drift Detection & Monitoring
